# RecordDiff: Full Two-Stage Model (build, smoke-test, train)

## 1 · Setup

In [ ]:
import os, time, math, json, numpy as np, torch
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)
if device == "cpu":
    print("WARNING: no GPU. Enable Runtime > Change runtime type > GPU for the real-data run.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Configuration

In [ ]:
# paths-
CACHE_NPZ = "..."
CKPT      = "..."
SYNTH_OUT = "..."

D_C     = 8
D_Z     = 32
D_E     = 128
D_RNN   = 128
D_COMB  = 128
H_MASK  = 256
H_DEN   = 256
N_DIFF  = 100

EPOCHS   = (15, 8, 15)
BATCH    = 256
LR       = 1e-3
BETA_MAX = 1.0
LAMBDA_M = 1.0
FREE_BITS = 0.02
N_GEN    = 3000
print("config loaded | latent d_z =", D_Z, "| schedule", EPOCHS, "| diffusion steps", N_DIFF)

## 3 · Model core

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# utils
def timestep_embedding(t, dim):
    """Sinusoidal embedding of diffusion step t:(B,) -> (B,dim)."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / max(half, 1))
    args = t.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        emb = F.pad(emb, (0, 1))
    return emb


def causal_hist(m):
    """m:(B,T,V) {0,1} -> (B,T,2V): [count-before-t / T, steps-since-last-obs-before-t / T].
    Strictly causal (uses only t' < t), matching the incremental generation-time update."""
    B, T, V = m.shape
    csum = torch.cumsum(m, dim=1)
    count_excl = csum - m
    idx = torch.arange(T, device=m.device).view(1, T, 1).float().expand(B, T, V)
    obs_pos = torch.where(m > 0.5, idx, torch.full_like(m, -1.0))
    shifted = torch.cat([torch.full((B, 1, V), -1.0, device=m.device), obs_pos[:, :-1, :]], dim=1).contiguous()
    last_obs_excl = torch.cummax(shifted, dim=1).values
    dt = idx - last_obs_excl
    return torch.cat([count_excl / T, dt / T], dim=-1)


class DiffusionSchedule:
    """Standard DDPM linear-beta schedule with precomputed coefficients."""
    def __init__(self, n_steps=100, beta_start=1e-4, beta_end=2e-2):
        betas = torch.linspace(beta_start, beta_end, n_steps)
        alphas = 1.0 - betas
        abar = torch.cumprod(alphas, dim=0)
        abar_prev = torch.cat([torch.ones(1), abar[:-1]])
        self.n_steps = n_steps
        self.betas = betas
        self.alphas = alphas
        self.abar = abar
        self.sqrt_abar = torch.sqrt(abar)
        self.sqrt_one_minus_abar = torch.sqrt(1.0 - abar)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
        self.posterior_var = betas * (1.0 - abar_prev) / (1.0 - abar)

    def to(self, device):
        for k, v in list(self.__dict__.items()):
            if torch.is_tensor(v):
                setattr(self, k, v.to(device))
        return self

    def q_sample(self, x0, t, noise):
        sa = self.sqrt_abar[t].view(-1, 1, 1)
        soma = self.sqrt_one_minus_abar[t].view(-1, 1, 1)
        return sa * x0 + soma * noise


class Denoiser(nn.Module):
    """Predicts diffusion noise per timestep: eps_theta(x_noisy, cond, t). Applied vectorised over T."""
    def __init__(self, V, d_cond, d_temb=64, hidden=256):
        super().__init__()
        self.d_temb = d_temb
        self.net = nn.Sequential(
            nn.Linear(V + d_cond + d_temb, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, V),
        )

    def forward(self, x, cond, t):
        temb = timestep_embedding(t, self.d_temb)
        temb = temb[:, None, :].expand(-1, x.shape[1], -1)
        return self.net(torch.cat([x, cond, temb], dim=-1))

# model
class RecordDiff(nn.Module):
    def __init__(self, V, d_c=8, d_z=32, d_e=128, d_rnn=128, d_comb=128,
                 hidden_mask=256, hidden_den=256, d_temb=64, n_diff=100):
        super().__init__()
        self.V, self.d_c, self.d_z = V, d_c, d_z
        self.diff = DiffusionSchedule(n_diff)

        self.emb = nn.Sequential(nn.Linear(2 * V, d_e), nn.SiLU(), nn.Linear(d_e, d_e))
        self.bigru = nn.GRU(d_e, d_rnn, batch_first=True, bidirectional=True)
        self.comb_z = nn.Linear(d_z, d_comb)
        self.comb_g = nn.Linear(2 * d_rnn, d_comb)
        self.q_mu = nn.Linear(d_comb, d_z)
        self.q_ls = nn.Linear(d_comb, d_z)
        self.p0_mu = nn.Linear(d_c, d_z)
        self.p0_ls = nn.Linear(d_c, d_z)
        self.tr_body = nn.Sequential(nn.Linear(d_z + d_c, d_comb), nn.SiLU())
        self.tr_mu = nn.Linear(d_comb, d_z)
        self.tr_ls = nn.Linear(d_comb, d_z)
        self.mask_head = nn.Sequential(
            nn.Linear(d_z + d_c + 2 * V, hidden_mask), nn.SiLU(),
            nn.Linear(hidden_mask, hidden_mask), nn.SiLU(),
            nn.Linear(hidden_mask, V))
        self.d_cond = d_z + d_c + V + 2 * V
        self.denoiser = Denoiser(V, self.d_cond, d_temb, hidden_den)
        self.register_buffer("norm_mean", torch.zeros(V))
        self.register_buffer("norm_std", torch.ones(V))

    def params_encoder(self):
        mods = [self.emb, self.bigru, self.comb_z, self.comb_g, self.q_mu, self.q_ls,
                self.p0_mu, self.p0_ls, self.tr_body, self.tr_mu, self.tr_ls]
        return [p for mod in mods for p in mod.parameters()]

    def params_value(self):
        return list(self.denoiser.parameters())

    def params_mask(self):
        return list(self.mask_head.parameters())

    def set_normalizer(self, mean, std):
        self.norm_mean.data = mean.to(self.norm_mean.device)
        self.norm_std.data = std.clamp(min=1e-3).to(self.norm_std.device)

    def _prior_step(self, z_prev, c, t):
        if t == 0:
            return self.p0_mu(c), self.p0_ls(c)
        body = self.tr_body(torch.cat([z_prev, c], dim=-1))
        return z_prev + self.tr_mu(body), self.tr_ls(body)

    def infer(self, y, m, c):
        """Amortised posterior over z_{1:T}; returns Z and KL(q||p) (free-bits, mean over B,T)."""
        B, T, V = y.shape
        e = self.emb(torch.cat([y, m], dim=-1))
        g, _ = self.bigru(e)
        z_prev = torch.zeros(B, self.d_z, device=y.device)
        Zs, KLs = [], []
        for t in range(T):
            hc = 0.5 * (torch.tanh(self.comb_z(z_prev)) + self.comb_g(g[:, t, :]))
            mu_q, ls_q = self.q_mu(hc), self.q_ls(hc)
            mu_p, ls_p = self._prior_step(z_prev, c, t)
            std_q = F.softplus(ls_q) + 1e-4
            std_p = F.softplus(ls_p) + 1e-4
            z = mu_q + std_q * torch.randn_like(std_q)
            kl = (torch.log(std_p / std_q)
                  + (std_q ** 2 + (mu_q - mu_p) ** 2) / (2 * std_p ** 2) - 0.5)
            Zs.append(z); KLs.append(kl)
            z_prev = z
        Z = torch.stack(Zs, dim=1)
        KL = torch.stack(KLs, dim=1)
        return Z, KL

    def losses(self, y, m, c, free_bits=0.02):
        """y assumed already normalised; m in {0,1}; c:(B,d_c). Returns loss dict."""
        B, T, V = y.shape
        hist = causal_hist(m)
        Z, KL = self.infer(y, m, c)
        c_seq = c[:, None, :].expand(-1, T, -1)

        mask_logits = self.mask_head(torch.cat([Z, c_seq, hist], dim=-1))
        L_mask = F.binary_cross_entropy_with_logits(mask_logits, m)

        x0 = y
        tau = torch.randint(0, self.diff.n_steps, (B,), device=y.device)
        noise = torch.randn_like(x0)
        x_noisy = self.diff.q_sample(x0, tau, noise)
        cond = torch.cat([Z, c_seq, m, hist], dim=-1)
        eps_pred = self.denoiser(x_noisy, cond, tau)
        se = (eps_pred - noise) ** 2
        L_value = (se * m).sum() / m.sum().clamp(min=1.0)

        KL_fb = torch.clamp(KL, min=free_bits).sum(-1).mean()
        return {"L_value": L_value, "L_mask": L_mask, "KL": KL_fb}

    # generation
    @torch.no_grad()
    def _ddpm_sample(self, cond):
        """Reverse DDPM for a single timestep. cond:(n,d_cond) -> x:(n,V) (normalised)."""
        n = cond.shape[0]
        dev = cond.device
        x = torch.randn(n, 1, self.V, device=dev)
        cond1 = cond[:, None, :]
        for i in reversed(range(self.diff.n_steps)):
            tau = torch.full((n,), i, dtype=torch.long, device=dev)
            eps = self.denoiser(x, cond1, tau)
            mean = self.diff.sqrt_recip_alphas[i] * (
                x - self.diff.betas[i] / self.diff.sqrt_one_minus_abar[i] * eps)
            if i > 0:
                x = mean + torch.sqrt(self.diff.posterior_var[i]) * torch.randn_like(x)
            else:
                x = mean
        return x[:, 0, :]

    @torch.no_grad()
    def generate(self, n, c, T, denorm=True):
        """Ancestral sample of recorded reality. Returns (Y, M), Y denormalised if denorm=True."""
        dev = c.device
        V = self.V
        count_run = torch.zeros(n, V, device=dev)
        last_obs = -torch.ones(n, V, device=dev)
        z_prev = torch.zeros(n, self.d_z, device=dev)
        M = torch.zeros(n, T, V, device=dev)
        Y = torch.zeros(n, T, V, device=dev)
        for t in range(T):
            mu_p, ls_p = self._prior_step(z_prev, c, t)
            z = mu_p + (F.softplus(ls_p) + 1e-4) * torch.randn_like(mu_p)
            hist = torch.cat([count_run / T, (t - last_obs) / T], dim=-1)
            m_t = torch.bernoulli(torch.sigmoid(self.mask_head(torch.cat([z, c, hist], dim=-1))))
            x_t = self._ddpm_sample(torch.cat([z, c, m_t, hist], dim=-1))
            if denorm:
                x_t = x_t * self.norm_std + self.norm_mean
            M[:, t, :] = m_t
            Y[:, t, :] = m_t * x_t
            last_obs = torch.where(m_t > 0.5, torch.full_like(last_obs, float(t)), last_obs)
            count_run = count_run + m_t
            z_prev = z
        return Y, M

# data helpers
def fit_normalizer(y, m):
    """Per-variable mean/std over observed (m==1) entries. y,m:(N,T,V) tensors."""
    V = y.shape[-1]
    mean = torch.zeros(V); std = torch.ones(V)
    for v in range(V):
        vals = y[:, :, v][m[:, :, v] > 0.5]
        if vals.numel() > 10:
            mean[v] = vals.mean()
            std[v] = vals.std().clamp(min=1e-3)
    return mean, std


def normalize(y, m, mean, std):
    yn = (y - mean) / std
    return yn * m


def set_requires_grad(params, flag):
    for p in params:
        p.requires_grad_(flag)


def train_recorddiff(model, y, m, c, epochs=(8, 4, 8), batch=256, lr=1e-3,
                     beta_max=1.0, lambda_m=1.0, free_bits=0.02, val_frac=0.1,
                     clip=5.0, device="cpu", seed=0, verbose=True):
    """3-phase amortised-VI training.
        Phase 1 (representation): encoder + value diffusion + prior,  loss = L_value + beta*KL
        Phase 2 (policy):         freeze the above, train mask head,   loss = L_mask
        Phase 3 (joint):          everything,  loss = L_value + lambda_m*L_mask + beta_max*KL
    y is RAW (normalised internally via model.norm_*). Returns loss history."""
    torch.manual_seed(seed)
    model.to(device); model.diff.to(device)
    N = y.shape[0]
    perm = torch.randperm(N)
    n_val = int(N * val_frac)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    e1, e2, e3 = epochs
    total = e1 + e2 + e3
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mean, std = model.norm_mean.cpu(), model.norm_std.cpu()
    hist = {"phase": [], "L_value": [], "L_mask": [], "KL": [], "val_total": []}

    def run_batches(idx, train=True):
        agg = {"L_value": 0.0, "L_mask": 0.0, "KL": 0.0, "tot": 0.0, "nb": 0}
        order = idx[torch.randperm(len(idx))] if train else idx
        for s in range(0, len(order), batch):
            bi = order[s:s + batch]
            yb = normalize(y[bi], m[bi], mean, std).to(device)
            mb = m[bi].to(device)
            cb = c[bi].to(device)
            out = model.losses(yb, mb, cb, free_bits=free_bits)
            if phase == 1:
                loss = out["L_value"] + beta * out["KL"]
            elif phase == 2:
                loss = out["L_mask"]
            else:
                loss = out["L_value"] + lambda_m * out["L_mask"] + beta_max * out["KL"]
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip); opt.step()
            for k in ("L_value", "L_mask", "KL"):
                agg[k] += float(out[k].detach())
            agg["tot"] += float(loss.detach()); agg["nb"] += 1
        for k in ("L_value", "L_mask", "KL", "tot"):
            agg[k] /= max(agg["nb"], 1)
        return agg

    for ep in range(total):
        if ep < e1:
            phase = 1; beta = beta_max * min(1.0, (ep + 1) / max(e1, 1))
            set_requires_grad(model.params_encoder(), True)
            set_requires_grad(model.params_value(), True)
            set_requires_grad(model.params_mask(), False)
        elif ep < e1 + e2:
            phase = 2; beta = beta_max
            set_requires_grad(model.params_encoder(), False)
            set_requires_grad(model.params_value(), False)
            set_requires_grad(model.params_mask(), True)
        else:
            phase = 3; beta = beta_max
            set_requires_grad(model.params_encoder(), True)
            set_requires_grad(model.params_value(), True)
            set_requires_grad(model.params_mask(), True)

        model.train(); tr = run_batches(tr_idx, train=True)
        model.eval()
        with torch.no_grad():
            va = run_batches(val_idx, train=False) if n_val > 0 else tr
        hist["phase"].append(phase)
        hist["L_value"].append(tr["L_value"]); hist["L_mask"].append(tr["L_mask"])
        hist["KL"].append(tr["KL"]); hist["val_total"].append(va["tot"])
        if verbose:
            print(f"ep {ep:3d} | phase {phase} | "
                  f"L_value {tr['L_value']:.4f}  L_mask {tr['L_mask']:.4f}  KL {tr['KL']:.4f} "
                  f"| val_total {va['tot']:.4f}")
    return hist

# synthetic
def make_synthetic(n=512, T=12, V=6, seed=0, device="cpu"):
    """Toy MNAR data: latent severity drives both values and (informatively) the mask."""
    g = torch.Generator().manual_seed(seed)
    sev = torch.rand(n, 1, 1, generator=g)
    base = torch.randn(n, 1, V, generator=g)
    drift = torch.linspace(0, 1, T).view(1, T, 1) * (sev - 0.5) * 4.0
    x = base + drift + 0.3 * torch.randn(n, T, V, generator=g)
    logit = -0.5 + 2.0 * sev + 0.6 * (x > 1.0).float() + 0.4 * torch.randn(n, T, V, generator=g)
    m = torch.bernoulli(torch.sigmoid(logit), generator=g)
    y = x * m
    c = torch.cat([sev.view(n, 1), torch.randn(n, 7, generator=g)], dim=-1)
    return y.to(device), m.to(device), c.to(device)


## 4 · Smoke test on synthetic data

In [ ]:
ys, ms, cs = make_synthetic(n=512, T=12, V=6, seed=0)
mean_s, std_s = fit_normalizer(ys, ms)
sm = RecordDiff(V=6, d_c=8, d_z=8, d_e=32, d_rnn=32, d_comb=32,
                hidden_mask=64, hidden_den=64, d_temb=32, n_diff=20).to(device)
sm.diff.to(device)
sm.set_normalizer(mean_s, std_s)
hist_s = train_recorddiff(sm, ys, ms, cs, epochs=(3, 2, 3), batch=128, lr=2e-3,
                          beta_max=1.0, device=device, seed=0, verbose=True)

assert all(np.isfinite(hist_s["L_value"])) and all(np.isfinite(hist_s["L_mask"])), "non-finite loss"
sm.eval(); sm.diff.to(device)
Yg, Mg = sm.generate(n=64, c=torch.zeros(64, 8, device=device), T=12)
assert tuple(Yg.shape) == (64, 12, 6) and tuple(Mg.shape) == (64, 12, 6), "bad gen shape"
uniq = set(torch.unique(Mg).tolist()); assert uniq <= {0.0, 1.0}, f"mask not binary: {uniq}"
leak = (Yg * (1 - Mg)).abs().max().item(); assert leak == 0.0, f"values leak where unobserved: {leak}"
print("\nSMOKE OK — losses finite, generation shapes/values valid.")
print(f"  gen mask density {Mg.mean().item():.3f} | real(synth) mask density {ms.mean().item():.3f}")

## 5 · Load the cached cohort


In [ ]:
d = np.load(CACHE_NPZ, allow_pickle=True)
m_np, y_np = d["m"].astype(np.float32), d["y"].astype(np.float32)
VAR_NAMES = list(d["var_names"]); VAR_CLASS = d["var_class"]
N, T, V = m_np.shape
m = torch.from_numpy(m_np); y = torch.from_numpy(y_np)
c   = torch.from_numpy(d["c"].astype(np.float32))
D_C = c.shape[1]
print("covariates:", list(d["cov_names"]))
for v in range(V):
    vals = y[:, :, v][m[:, :, v] > 0.5]
    if vals.numel() > 200:
        s = vals[torch.randperm(vals.numel())[:200000]]
        lo, hi = torch.quantile(s, torch.tensor([0.005, 0.995]))
        y[:, :, v] = torch.clamp(y[:, :, v], float(lo), float(hi)) * m[:, :, v]
print("winsorized values per variable")

mean, std = fit_normalizer(y, m)
print(f"cohort: N={N:,}  T={T}  V={V}  | overall mask density {m.mean().item():.4f}")
print("per-variable observed rate (first 12):")
dens = m.mean(dim=(0, 1))
for i in range(min(12, V)):
    print(f"  {VAR_NAMES[i]:16s} [{VAR_CLASS[i]:9s}] rate={dens[i].item():.3f}")

## 6 · Train (3-phase schedule) and checkpoint


In [ ]:
model = RecordDiff(V=V, d_c=D_C, d_z=D_Z, d_e=D_E, d_rnn=D_RNN, d_comb=D_COMB,
                   hidden_mask=H_MASK, hidden_den=H_DEN, d_temb=64, n_diff=N_DIFF).to(device)
model.diff.to(device)
model.set_normalizer(mean, std)
t0 = time.time()
hist = train_recorddiff(model, y, m, c, epochs=EPOCHS, batch=BATCH, lr=LR,
                        beta_max=BETA_MAX, lambda_m=LAMBDA_M, free_bits=FREE_BITS,
                        device=device, seed=0, verbose=True)
print(f"\ntrained in {(time.time()-t0)/60:.1f} min")

torch.save({"state_dict": model.state_dict(),
            "config": dict(V=V, d_c=D_C, d_z=D_Z, d_e=D_E, d_rnn=D_RNN, d_comb=D_COMB,
                           hidden_mask=H_MASK, hidden_den=H_DEN, d_temb=64, n_diff=N_DIFF),
            "var_names": VAR_NAMES, "var_class": VAR_CLASS,
            "norm_mean": mean, "norm_std": std}, CKPT)
print("checkpoint saved ->", CKPT)

In [ ]:
# training curves
ep = np.arange(len(hist["L_value"]))
b1, b2 = EPOCHS[0], EPOCHS[0] + EPOCHS[1]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, key, ttl in zip(ax, ["L_value", "L_mask", "KL"],
                       ["value diffusion loss", "mask BCE loss", "KL"]):
    a.plot(ep, hist[key]); a.set_title(ttl); a.set_xlabel("epoch")
    a.axvline(b1 - 0.5, ls="--", c="gray", lw=1); a.axvline(b2 - 0.5, ls="--", c="gray", lw=1)
ax[0].plot(ep, hist["val_total"], alpha=0.5, label="val total"); ax[0].legend()
plt.tight_layout(); plt.show()

## 7 · Generate synthetic records and sanity-check fidelity

In [ ]:
model.eval(); model.diff.to(device)
t0 = time.time()
with torch.no_grad():
    idx   = torch.randint(0, N, (N_GEN,))
    c_gen = c[idx].to(device)
    Yg, Mg = model.generate(n=N_GEN, c=c_gen, T=T)
Yg, Mg = Yg.cpu(), Mg.cpu()
print(f"generated {N_GEN} records in {time.time()-t0:.0f}s | shape {tuple(Yg.shape)}")

real_dens = m.mean(dim=(0, 1)); synth_dens = Mg.mean(dim=(0, 1))
def obs_mean(Y, M, v):
    vals = Y[:, :, v][M[:, :, v] > 0.5]
    return float(vals.mean()) if vals.numel() > 0 else float("nan")

print(f"\noverall mask density  real {m.mean().item():.4f}  synth {Mg.mean().item():.4f}")
print(f"\n{'variable':16s} {'class':9s} | {'dens_real':>9s} {'dens_synth':>10s} | {'mean_real':>9s} {'mean_synth':>10s}")
print("-" * 78)
for v in range(V):
    print(f"{VAR_NAMES[v]:16s} {VAR_CLASS[v]:9s} | {real_dens[v].item():9.3f} {synth_dens[v].item():10.3f} "
          f"| {obs_mean(y, m, v):9.2f} {obs_mean(Yg, Mg, v):10.2f}")

plt.figure(figsize=(5, 5))
col = {"protocol": "tab:blue", "acuity": "tab:orange", "triggered": "tab:red"}
for v in range(V):
    plt.scatter(real_dens[v], synth_dens[v], c=col.get(str(VAR_CLASS[v]), "gray"), s=20)
lims = [0, max(real_dens.max().item(), synth_dens.max().item()) * 1.05]
plt.plot(lims, lims, "k--", lw=1); plt.xlabel("real mask density"); plt.ylabel("synth mask density")
plt.title("per-variable mask density: real vs synthetic"); plt.tight_layout(); plt.show()

np.savez_compressed(SYNTH_OUT, m=Mg.numpy().astype(np.int8), y=Yg.numpy().astype(np.float32),
                    var_names=np.array(VAR_NAMES, dtype=object), var_class=VAR_CLASS)
print("synthetic sample saved ->", SYNTH_OUT)